# 🎙️ Nepali ASR — FastAPI Server (Run in Kaggle)

This notebook loads the IndicConformer model and exposes it as a **FastAPI** REST API tunnelled publicly via **ngrok**.
Your local PC sends audio → Kaggle transcribes it → returns Nepali text.

**Steps:**
1. Select **GPU** accelerator: `Settings → Accelerator → GPU T4 x2`
2. Add your ngrok token as a Kaggle Secret named `ngrok_auth` (Settings → Secrets)
3. Add your `.nemo` model file as a Kaggle Dataset and attach it to this notebook
4. Run all cells top to bottom
5. Copy the printed public URL → paste into `asr_client.py` on your PC

**API Endpoints after startup:**

| Method | Path | Description |
|--------|------|-------------|
| `GET`  | `/health` | Server + GPU status |
| `POST` | `/transcribe` | Upload audio → Nepali text (CTC default) |
| `POST` | `/transcribe?decoder=rnnt` | Use RNNT decoder (more accurate, slower) |
| `GET`  | `/docs` | Auto-generated Swagger UI |

## ⚙️ Section 1 — Install Dependencies

In [ ]:
# Pin huggingface_hub (required by AI4Bharat NeMo fork)
!pip install -q "huggingface_hub==0.15.1"
print("✅ huggingface_hub pinned")

In [ ]:
# Core ML + audio dependencies
!pip install -q \
    torchaudio \
    pytorch-lightning==2.0.9 \
    omegaconf \
    hydra-core \
    scipy soundfile librosa sentencepiece editdistance jiwer packaging Cython
print("✅ Core ML + audio dependencies installed")

In [ ]:
# Clone and install AI4Bharat's NeMo fork
import os
if not os.path.exists("/kaggle/working/NeMo"):
    !git clone -q https://github.com/AI4Bharat/NeMo.git /kaggle/working/NeMo
    print("✅ Cloned AI4Bharat NeMo")
else:
    print("✅ NeMo already cloned")

%cd /kaggle/working/NeMo
!pip install -q -e ".[asr]" 2>&1 | tail -5
print("✅ NeMo installed")

In [ ]:
# FastAPI server dependencies (replaces Flask)
!pip install -q fastapi "uvicorn[standard]" python-multipart pyngrok
print("✅ FastAPI + uvicorn + pyngrok installed")

## 🔑 Section 2 — ngrok Auth Token

Sign up free at https://ngrok.com → Dashboard → **Your Authtoken**.
Then add it as a Kaggle Secret: `Notebook Settings → Secrets → Add` with the name `ngrok_auth`.

In [ ]:
# ── Read ngrok token from Kaggle Secrets ───────────────────────
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
NGROK_AUTH_TOKEN = user_secrets.get_secret("ngrok_auth")
# ──────────────────────────────────────────────────────────────

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTH_TOKEN
print("✅ ngrok configured")

## 🤖 Section 3 — Load the ASR Model

Make sure your `.nemo` model file is attached as a Kaggle Dataset.
Go to `Notebook Settings → Add Data` and attach the dataset that contains `indicconformer_stt_ne_hybrid_ctc_rnnt_large_v2.nemo`.
The file will be available at `/kaggle/input/<your-dataset-name>/`.

In [ ]:
import numpy as np
if not hasattr(np, 'sctypes'):
    np.sctypes = {
        'int': [np.int8, np.int16, np.int32, np.int64],
        'uint': [np.uint8, np.uint16, np.uint32, np.uint64],
        'float': [np.float16, np.float32, np.float64],
        'complex': [np.complex64, np.complex128],
        'others': [bool, object, bytes, str]
    }
    print("✅ np.sctypes patched for NumPy 2.x")

In [ ]:
import os
import torch
import nemo.collections.asr as nemo_asr

# ── Update the dataset name below to match your attached Kaggle dataset ──
MODEL_DIR = "/kaggle/input/nemo-files"   # ← change 'nemo-files' to your dataset name
NEMO_FILE = os.path.join(MODEL_DIR, "indicconformer_stt_ne_hybrid_ctc_rnnt_large_v2.nemo")

assert os.path.exists(NEMO_FILE), (
    f"Model not found at {NEMO_FILE}. "
    "Attach the dataset containing the .nemo file via Notebook Settings → Add Data."
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading model on: {device}")

model = nemo_asr.models.EncDecHybridRNNTCTCModel.restore_from(
    restore_path=NEMO_FILE,
    map_location=device,
)
model = model.to(device)
model.eval()
model.freeze()

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded — {total_params:,} params on {device}")

## 🌐 Section 4 — Define the FastAPI App

In [ ]:
import uuid
import torchaudio
from typing import Literal

from fastapi import FastAPI, File, HTTPException, Query, UploadFile
from fastapi.responses import JSONResponse

app = FastAPI(
    title="Nepali ASR API",
    description="IndicConformer-based Nepali speech recognition running on Kaggle GPU",
    version="1.0.0",
)

UPLOAD_DIR = "/tmp/asr_uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

ALLOWED_EXTENSIONS = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}


# ── Helper ─────────────────────────────────────────────────────────────────
def preprocess_audio(input_path: str, target_sr: int = 16000) -> str:
    """Convert any audio file to 16 kHz mono WAV as required by the model."""
    base, _ = os.path.splitext(input_path)
    output_path = f"{base}_ready.wav"

    wav, sr = torchaudio.load(input_path)

    if wav.shape[0] > 1:            # stereo → mono
        wav = wav.mean(dim=0, keepdim=True)

    if sr != target_sr:             # resample to 16 kHz
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)
        wav = resampler(wav)

    torchaudio.save(output_path, wav, target_sr)
    return output_path


# ── GET /health ─────────────────────────────────────────────────────────────
@app.get("/health", summary="Health check")
async def health():
    """Returns server status and GPU information."""
    return {
        "status": "ok",
        "device": str(device),
        "cuda_available": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }


# ── POST /transcribe ────────────────────────────────────────────────────────
@app.post("/transcribe", summary="Transcribe Nepali audio")
async def transcribe(
    audio: UploadFile = File(
        ..., description="Audio file to transcribe (wav / mp3 / flac / ogg / m4a)"
    ),
    decoder: Literal["ctc", "rnnt"] = Query(
        default="ctc",
        description="ctc = faster greedy decoding  |  rnnt = more accurate beam search",
    ),
):
    """
    Send an audio file and receive the Nepali transcription.

    - **audio** : multipart file upload (wav, mp3, flac, ogg, m4a)
    - **decoder**: `ctc` (default, fast) or `rnnt` (slower, more accurate)
    """

    # ── Validate file extension ────────────────────────────────
    ext = os.path.splitext(audio.filename or "")[-1].lower()
    if ext not in ALLOWED_EXTENSIONS:
        raise HTTPException(
            status_code=415,
            detail=f"Unsupported file type '{ext}'. Allowed: {sorted(ALLOWED_EXTENSIONS)}",
        )

    uid      = uuid.uuid4().hex
    raw_path = os.path.join(UPLOAD_DIR, f"{uid}{ext}")
    ready_path = os.path.join(UPLOAD_DIR, f"{uid}_ready.wav")

    try:
        # ── Save the uploaded bytes ────────────────────────────
        contents = await audio.read()
        with open(raw_path, "wb") as f:
            f.write(contents)

        # ── Preprocess: any format → 16 kHz mono WAV ──────────
        ready_path = preprocess_audio(raw_path)

        # ── Run inference on GPU ───────────────────────────────
        model.cur_decoder = decoder
        results = model.transcribe(
            audio=[ready_path],
            batch_size=1,
            language_id="ne",
            logprobs=False,
            verbose=False,
        )

        # results[0] may be a string or a Hypothesis object
        text = results[0]
        if hasattr(text, "text"):
            text = text.text
        text = str(text).strip()

        return JSONResponse(content={
            "transcription": text,
            "decoder": decoder,
            "filename": audio.filename,
        })

    except HTTPException:
        raise  # re-raise FastAPI HTTP errors as-is
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

    finally:
        # Always clean up temp files
        for path in [raw_path, ready_path]:
            if os.path.exists(path):
                os.remove(path)


print("✅ FastAPI app defined — endpoints: /health  /transcribe  /docs")

## 🚀 Section 5 — Start uvicorn + ngrok Tunnel

Run this cell last. It will print your **public URL** — copy it into `asr_client.py` on your PC.

In [ ]:
import threading
import uvicorn
from pyngrok import ngrok

PORT = 8000

# Open the ngrok tunnel
tunnel = ngrok.connect(PORT)
public_url = tunnel.public_url if hasattr(tunnel, "public_url") else str(tunnel)

print("\n" + "=" * 64)
print("  🚀  Nepali ASR — FastAPI Server is LIVE")
print("=" * 64)
print(f"  📡  Public URL    : {public_url}")
print(f"  🩺  Health check  : {public_url}/health")
print(f"  🎙️   Transcribe    : POST {public_url}/transcribe")
print(f"  📖  Swagger docs  : {public_url}/docs")
print("=" * 64)
print("  👉  Copy the Public URL → paste into asr_client.py on your PC")
print("=" * 64 + "\n")

# Run uvicorn in a daemon thread so the cell stays alive without blocking Kaggle
config = uvicorn.Config(
    app=app,
    host="0.0.0.0",
    port=PORT,
    log_level="warning",   # change to 'info' if you want request logs
)
server = uvicorn.Server(config)

thread = threading.Thread(target=server.run, daemon=True)
thread.start()

print("⏳ Server running — keep this cell alive.")
print("   Stop: Runtime → Interrupt execution  (closes ngrok tunnel too).")